# Lab 02-01: GAN on MNIST (Colab T4 Optimized)

## Objective
Train a basic GAN Model to generate synthetic handwritten digits (MNIST).

## Optimizations for Colab T4
- **Batch Size**: Increased to 128 for better GPU utilization.
- **Data Loading**: `num_workers=2`, `pin_memory=True`.
- **CUDNN Benchmark**: Enabled for faster training.
- **Progress Bars**: Added `tqdm` for visibility.

## Components
1. **Pre-trained Classifier**: Train a simple classifier on MNIST for evaluation.
2. **GAN Implementation**: Generator & Discriminator.
3. **Training**: Train for 50 epochs.
4. **Evaluation**: Generate images and classify them.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import save_image, make_grid
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os
from tqdm.auto import tqdm

# --- Configuration (T4 Optimized) ---
DATASET_CHOICE = 'mnist'
EPOCHS = 50 
BATCH_SIZE = 128  # Increased for T4
NOISE_DIM = 100
LR = 0.0002
SAVE_INTERVAL = 5
IMG_SIZE = 28
CHANNELS = 1

# Device Config with Optimizations
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    # Optimize for fixed input size
    torch.backends.cudnn.benchmark = True
else:
    print("⚠️  No GPU detected. Training will be slow.")

# Create directories
os.makedirs("generated_samples/01", exist_ok=True)
os.makedirs("final_generated_images/01", exist_ok=True)
os.makedirs("models/01", exist_ok=True)
print("Directories created.")

In [ ]:
# --- Step 1: Train a Classifier (for later evaluation) ---
class SimpleClassifier(nn.Module):
    def __init__(self):
        super(SimpleClassifier, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)
        self.pool = nn.MaxPool2d(2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def train_classifier():
    print("Training Evaluation Classifier (1 epoch)...")
    transform_cls = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    ds = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform_cls)
    # Optimized DataLoader
    dl = DataLoader(ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
    
    model = SimpleClassifier().to(device)
    crit = nn.CrossEntropyLoss()
    opt = optim.Adam(model.parameters(), lr=0.001)
    
    model.train()
    for imgs, lbls in tqdm(dl, desc="Training Classifier"):
        imgs, lbls = imgs.to(device), lbls.to(device)
        opt.zero_grad()
        out = model(imgs)
        loss = crit(out, lbls)
        loss.backward()
        opt.step()
        
    print("Classifier trained.")
    torch.save(model.state_dict(), "models/01/mnist_classifier.pth")
    return model

if not os.path.exists("models/01/mnist_classifier.pth"):
    eval_classifier = train_classifier()
else:
    print("Loading existing classifier...")
    eval_classifier = SimpleClassifier().to(device)
    eval_classifier.load_state_dict(torch.load("models/01/mnist_classifier.pth"))
    eval_classifier.eval()

In [ ]:
# --- Step 2: GAN Setup ---

# Data Loading (Normalized to [-1, 1])
gan_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=gan_transform)
# Optimized DataLoader
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=2, pin_memory=True)

# 1. Generator
class Generator(nn.Module):
    def __init__(self, noise_dim, img_dim):
        super(Generator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024, img_dim),
            nn.Tanh() # Outputs -1 to 1
        )
    def forward(self, x):
        return self.net(x)

# 2. Discriminator
class Discriminator(nn.Module):
    def __init__(self, img_dim):
        super(Discriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(img_dim, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid() # Outputs 0 to 1
        )
    def forward(self, x):
        return self.net(x)

# Init Models
gen = Generator(NOISE_DIM, 28*28).to(device)
disc = Discriminator(28*28).to(device)

opt_g = optim.Adam(gen.parameters(), lr=LR)
opt_d = optim.Adam(disc.parameters(), lr=LR)
criterion = nn.BCELoss()

fixed_noise = torch.randn(25, NOISE_DIM).to(device) 

print("GAN Models initialized.")

In [ ]:
# --- Step 3: Training Loop ---
print(f"Starting Training for {EPOCHS} epochs...")

for epoch in range(EPOCHS):
    # progress bar for the epoch
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for batch_idx, (real, _) in pbar:
        real = real.view(-1, 784).to(device)
        curr_batch_size = real.shape[0]
        
        real_labels = torch.ones(curr_batch_size, 1).to(device)
        fake_labels = torch.zeros(curr_batch_size, 1).to(device)
        
        # --- Train Discriminator ---
        outputs = disc(real)
        d_loss_real = criterion(outputs, real_labels)
        real_score = outputs.mean().item()

        noise = torch.randn(curr_batch_size, NOISE_DIM).to(device)
        fake = gen(noise)
        outputs = disc(fake.detach())
        d_loss_fake = criterion(outputs, fake_labels)
        fake_score = outputs.mean().item()
        
        d_loss = (d_loss_real + d_loss_fake) / 2
        opt_d.zero_grad()
        d_loss.backward()
        opt_d.step()
        
        # --- Train Generator ---
        outputs = disc(fake)
        g_loss = criterion(outputs, real_labels)
        
        opt_g.zero_grad()
        g_loss.backward()
        opt_g.step()
        
        # Update progress bar
        pbar.set_postfix({
            'D_loss': f"{d_loss.item():.4f}", 
            'G_loss': f"{g_loss.item():.4f}",
            'Real_S': f"{real_score:.2f}",
            'Fake_S': f"{fake_score:.2f}"
        })
        
    # --- Epoch Logging & Saving ---
    if (epoch+1) % SAVE_INTERVAL == 0:
        with torch.no_grad():
            fake_images = gen(fixed_noise).reshape(-1, 1, 28, 28)
            save_image(fake_images, f"generated_samples/01/epoch_{epoch+1:02d}.png", nrow=5, normalize=True)
            print(f"  -> Saved sample for epoch {epoch+1}")

print("Training Complete.")

In [ ]:
# --- Step 4: Evaluation ---
print("Generating 100 final samples...")
gen.eval()
with torch.no_grad():
    # 1. Generate
    final_noise = torch.randn(100, NOISE_DIM).to(device)
    final_fake = gen(final_noise).reshape(-1, 1, 28, 28)
    
    # 2. Save
    save_image(final_fake, "final_generated_images/01/final_100_samples.png", nrow=10, normalize=True)
    
    # 3. Predict
    eval_classifier.to(device)
    eval_classifier.eval()
    
    outputs = eval_classifier(final_fake)
    _, predicted = torch.max(outputs.data, 1)
    
    # 4. Report
    preds = predicted.cpu().numpy()
    
    plt.figure(figsize=(10, 5))
    plt.hist(preds, bins=range(11), align='left', rwidth=0.8, color='skyblue', edgecolor='black')
    plt.xticks(range(10))
    plt.xlabel('Digit Class')
    plt.title('Distribution of Generated Digits')
    plt.grid(axis='y', alpha=0.5)
    plt.show()
    
    print("Predicted labels:", preds)
    unique, counts = np.unique(preds, return_counts=True)
    print(dict(zip(unique, counts)))